## Hypothesis: Among neurons that are strongly activated in both, real activations are larger than fake activations. 
## This tests magnitude suppression

In [19]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

REAL_CSV = "preds_of_64Neurons_denseLayer_test.csv"
FAKE_CSV = "[img+obj_labels_to_fakes]fake_activations(test).csv"

ID_COL = "filenames"   # set to None if you don't have an id/filename column
FIRE_FRAC = 0.80      # 80% of max real activation per neuron
ALPHA = 0.01


In [20]:
real_df = pd.read_csv(REAL_CSV)
fake_df = pd.read_csv(FAKE_CSV)
# fake_df["filenames"] = fake_df["filenames"].str.replace("fake_", "", regex=False)

if ID_COL is not None and ID_COL in real_df.columns and ID_COL in fake_df.columns:
    real_df = real_df.sort_values(ID_COL).reset_index(drop=True)
    fake_df = fake_df.sort_values(ID_COL).reset_index(drop=True)

    if not real_df[ID_COL].equals(fake_df[ID_COL]):
        raise ValueError("Real/Fake rows do not align by filename. Fix pairing first.")
else:
    if len(real_df) != len(fake_df):
        raise ValueError("Real/Fake have different row counts; cannot assume pairing.")


In [21]:
exclude = {ID_COL} if ID_COL is not None else set()

common_cols = [c for c in real_df.columns if c in fake_df.columns and c not in exclude]
neuron_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(real_df[c])]

print("Neuron columns found:", len(neuron_cols))
if len(neuron_cols) == 0:
    raise ValueError("No numeric neuron columns found in both CSVs.")


Neuron columns found: 64


In [22]:
real_mat = real_df[neuron_cols].to_numpy(dtype=float)
fake_mat = fake_df[neuron_cols].to_numpy(dtype=float)

max_real = real_mat.max(axis=0)          # shape: (num_neurons,)
thr = FIRE_FRAC * max_real               # per-neuron threshold

valid_neuron = max_real > 0              # neurons that ever fire in real
print("Neurons with max_real > 0:", valid_neuron.sum(), "/", len(valid_neuron))


Neurons with max_real > 0: 64 / 64


In [23]:
# Apply only on valid neurons
R = real_mat[:, valid_neuron]
F = fake_mat[:, valid_neuron]
T = thr[valid_neuron]

# co-activated mask: real >= thr AND fake >= thr
keep = (R >= T) & (F >= T)

kept_real = R[keep]
kept_fake = F[keep]

print("Total positions (images × valid_neurons):", R.size)
print("Kept positions (co-activated in both):   ", kept_real.size)


Total positions (images × valid_neurons): 50752
Kept positions (co-activated in both):    125


In [24]:
if kept_real.size < 20:
    raise ValueError(f"Too few co-activated pairs to test reliably: {kept_real.size}")

diff = kept_real - kept_fake

# Wilcoxon ignores zero diffs; be explicit
nz = diff != 0
if nz.sum() < 20:
    raise ValueError(f"Too few nonzero diffs after filtering: {nz.sum()}")

stat, p = wilcoxon(diff[nz], alternative="greater", zero_method="wilcox")

wins = np.sum(diff[nz] > 0)
losses = np.sum(diff[nz] < 0)
prop_real_gt_fake = wins / (wins + losses)
r_rb = (wins - losses) / (wins + losses)

print("=== Co-activated (>=80% in BOTH) Wilcoxon: real > fake ===")
print(f"Pairs kept (co-activated)      : {kept_real.size}")
print(f"Nonzero diffs used             : {nz.sum()}")
print(f"Median(real-fake)              : {np.median(diff[nz]):.6f}")
print(f"Mean(real-fake)                : {np.mean(diff[nz]):.6f}")
print(f"Prop(real>fake)                : {prop_real_gt_fake:.3f}")
print(f"Wilcoxon stat                  : {stat:.6e}")
print(f"p-value (one-sided, real>fake) : {p:.6e}")
print(f"rank-biserial r_rb             : {r_rb:.3f}  (positive favors real)")
print("Decision:", "REJECT H0" if p < ALPHA else "fail to reject H0", f"at alpha={ALPHA}")


=== Co-activated (>=80% in BOTH) Wilcoxon: real > fake ===
Pairs kept (co-activated)      : 125
Nonzero diffs used             : 125
Median(real-fake)              : 0.039240
Mean(real-fake)                : 0.056195
Prop(real>fake)                : 0.536
Wilcoxon stat                  : 4.375000e+03
p-value (one-sided, real>fake) : 1.405236e-01
rank-biserial r_rb             : 0.072  (positive favors real)
Decision: fail to reject H0 at alpha=0.01


## Hypothesis: For neurons that fire strongly in real images, fake images have fewer activated neurons than real images.

### Define firing using real-derived thresholds

 ### Keep only positions where real ≥ threshold

 ### Then check whether fake also fires at those positions

In [25]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

REAL_CSV = "preds_of_64Neurons_denseLayer_test.csv"
FAKE_CSV = "[img+obj_labels_to_fakes]fake_activations(test).csv"

ID_COL = "filenames"   # set to None if you don't have an id/filename column
FIRE_FRAC = 0.80      # 80% of max real activation per neuron
ALPHA = 0.01

In [26]:
real_df = pd.read_csv(REAL_CSV)
fake_df = pd.read_csv(FAKE_CSV)
# fake_df["filenames"] = fake_df["filenames"].str.replace("fake_", "", regex=False)


if ID_COL is not None and ID_COL in real_df.columns and ID_COL in fake_df.columns:
    real_df = real_df.sort_values(ID_COL).reset_index(drop=True)
    fake_df = fake_df.sort_values(ID_COL).reset_index(drop=True)

    if not real_df[ID_COL].equals(fake_df[ID_COL]):
        raise ValueError("Real/Fake rows do not align by filename after sorting.")
else:
    if len(real_df) != len(fake_df):
        raise ValueError("Real/Fake have different number of rows; cannot assume pairing.")


In [27]:
exclude = {ID_COL} if (ID_COL is not None and ID_COL in real_df.columns) else set()

common_cols = [c for c in real_df.columns if c in fake_df.columns and c not in exclude]
neuron_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(real_df[c])]

print("Using neuron columns:", len(neuron_cols))
if len(neuron_cols) != 64:
    print("WARNING: expected 64 neuron columns, found", len(neuron_cols))


Using neuron columns: 64


In [28]:
real_mat = real_df[neuron_cols].to_numpy(dtype=float)  # shape: (N_images, N_neurons)
fake_mat = fake_df[neuron_cols].to_numpy(dtype=float)

max_real = real_mat.max(axis=0)                         # per-neuron max over real images
thr = FIRE_FRAC * max_real                              # per-neuron threshold

valid_neuron = max_real > 0                             # ignore neurons that never fire in real
print("Neurons with max_real>0:", valid_neuron.sum(), "/", len(valid_neuron))


Neurons with max_real>0: 64 / 64


In [29]:
R = real_mat[:, valid_neuron]
F = fake_mat[:, valid_neuron]
T = thr[valid_neuron]

# real firing set per image (Variant A filter)
real_fire = (R >= T)

# counts
real_count = real_fire.sum(axis=1)

# fake count only among those real-firing neurons
fake_count = ((F >= T) & real_fire).sum(axis=1)

print("Example (first 10 images):")
print("real_count:", real_count[:10])
print("fake_count:", fake_count[:10])


Example (first 10 images):
real_count: [2 0 8 0 0 2 1 0 0 1]
fake_count: [0 0 0 0 0 2 1 0 0 0]


In [30]:
diff = real_count - fake_count

# Wilcoxon ignores zeros by default, but we’ll be explicit for reporting
nz = diff != 0
if nz.sum() < 10:
    raise ValueError(f"Too few nonzero differences for Wilcoxon: {nz.sum()}")

stat, p = wilcoxon(diff[nz], alternative="greater", zero_method="wilcox")

# simple sign-based effect size on image-level differences
wins = np.sum(diff[nz] > 0)
losses = np.sum(diff[nz] < 0)
r_rb = (wins - losses) / (wins + losses)

print("=== Variant A (image-level): fake activates fewer of the real-firing neurons ===")
print(f"Images paired                : {len(diff)}")
print(f"Nonzero diffs used            : {nz.sum()}")
print(f"Mean real_count               : {real_count.mean():.3f}")
print(f"Mean fake_count               : {fake_count.mean():.3f}")
print(f"Median(real_count - fake_count): {np.median(diff[nz]):.3f}")
print(f"Prop(real_count > fake_count) : {(diff[nz] > 0).mean():.3f}")
print(f"Wilcoxon stat                 : {stat:.6e}")
print(f"p-value (one-sided, greater)  : {p:.6e}")
print(f"Effect (sign-based r_rb)      : {r_rb:.3f}  (positive favors real)")
print("Decision:", "REJECT H0" if p < ALPHA else "fail to reject H0", f"at alpha={ALPHA}")


=== Variant A (image-level): fake activates fewer of the real-firing neurons ===
Images paired                : 793
Nonzero diffs used            : 264
Mean real_count               : 0.803
Mean fake_count               : 0.158
Median(real_count - fake_count): 1.000
Prop(real_count > fake_count) : 1.000
Wilcoxon stat                 : 3.498000e+04
p-value (one-sided, greater)  : 2.663170e-47
Effect (sign-based r_rb)      : 1.000  (positive favors real)
Decision: REJECT H0 at alpha=0.01
